# Application code sample — Readmission outcome construction

**Project:** MIMIC-IV recorded 30-day hospital readmission analysis.  
**Applicant:** Jessica Xu.

## Contribution and AI assistance
Python code was generated with AI assistance. My project work focuses on reviewing research definitions, investigating data inconsistencies, and documenting analytical decisions. This sample demonstrates the resulting analytical workflow; it is not a claim of independently authored Python code.

## Running this sample
Requires Python 3 and pandas, plus Jupyter/IPython for notebook display. Obtain authorized access to MIMIC-IV separately; no source data are included. Place admissions.csv and patients.csv in data/raw relative to the notebook working directory, or change DATA_DIR in the setup cell. Run the cohort notebook first, then the labeling notebook, using the same working directory. Outputs are written under sample_outputs; existing files there will be replaced when export cells run.

## Review status and privacy
These are reviewed Steps 2 and 3 from an ongoing revision of a previously completed analysis. Earlier model results use a different cohort and are not results of this revised pipeline. Selected aggregate outputs are preserved verbatim from the applicant-provided notebook execution. Patient-level example outputs, local-path outputs, and execution counts have been removed. The aggregate results are historical saved outputs, not results recomputed during application preparation. Setup paths were adapted for portability; the analysis was not rerun after that change. Running the notebooks requires restricted data and may produce sensitive local outputs, which should not be shared.

## Scope
Uses full admission histories to construct exact 30-day return windows, investigates overlapping encounters and mortality chronology, and exports labeled and audit tables.


# Step 3 — Construct the 30-Day Readmission Label

## Purpose

We construct the outcome variable for predicting a recorded hospital
readmission within 30 days after discharge.

Step 2 produced 528,551 eligible index admissions from 216,762 patients.
These are the starting prediction observations for label construction.
Any additional exclusions required for reliable labeling will be
documented in this notebook.

## Prediction Task and Target

- **Unit of observation:** One eligible index hospital admission.
- **Prediction point:** Discharge from the index admission.
- **Outcome window:** The 30 days following discharge.
- **Target:** Any qualifying recorded readmission, including planned
  and unplanned returns.
- **Patient linkage:** Index and subsequent admissions must belong
  to the same `subject_id`.
- **Admission identity:** A return must represent a distinct hospital
  admission, identified by `hadm_id`.

Elective admission type alone does not exclude either a starting
admission or a candidate return.

This target describes recorded hospital use. It does not establish
whether a return was unplanned, preventable, or caused by poor care.
Returns outside the available source data are not captured.

## Mathematical Framework

For index admission i:

- p_i is the patient identifier.
- d_i is the discharge timestamp.
- a_j is the admission timestamp of a distinct qualifying return
  for the same patient.
- Δ_ij = a_j − d_i is the elapsed time between discharge and return.

The positive outcome is defined by the existence of a qualifying return
within the 30-day window:

$$
Y_i = 1
\quad \text{if there exists a qualifying admission } j
\text{ within 30 days after } d_i.
$$

The exact lower time boundary and treatment of simultaneous encounters,
overlaps, and transfers will be specified before finalizing labels.

We compare exact elapsed time with a 30-day interval. We do not truncate
the interval to whole days before determining eligibility.

A missing subsequent admission does not, by itself, establish complete
30-day follow-up. The interpretation of negative labels and any
follow-up restrictions will be documented separately.

## Pipeline Design Principles

### Separate Prediction Observations from Outcome Events

The Step 2 cohort determines which admissions receive labels.
The source admission history provides candidate subsequent events.

An admission excluded as a starting observation may still count as
a return after an earlier eligible discharge.

### Preserve Traceability

Labels remain linked to `subject_id` and the index `hadm_id`.
Any selected return event and its timing are retained for validation
and audit, separately from the model's predictors.

### Keep Future Information Out of Predictors

Future admission information is used only to construct and validate
the outcome.

Variables such as the next admission timestamp and time to readmission
must not enter the feature matrix.

For the final labeled observation set S:

$$
X_i = \text{features available by discharge},
\qquad
Y_i = \text{outcome observed after discharge}.
$$

Later notebooks construct X, split observations by patient, and fit
a model that estimates:

$$
\hat{p}_i = f_{\theta}(X_i).
$$

### Document Changes to the Observation Set

We preserve index admission identifiers throughout label construction.
Any additional exclusion must have an explicit reason and a recorded
count. We do not silently drop admissions during matching or merging.

## Review Sequence

1. Locate the input files and load both admission tables.
2. Validate identifiers and timestamps.
3. Examine admission sequences, overlaps, and simultaneous encounters.
4. Define qualifying return events and exact time boundaries.
5. Address follow-up limitations and death after discharge.
6. Construct and validate labels.
7. Summarize label counts and any additional exclusions.
8. Export the labeled data for feature engineering.

## Expected Output

The output will contain eligible index admissions and their
`label_readmit_30d` outcome, together with the identifiers and audit
information needed to verify label construction.

The final labeled sample size and readmission prevalence will be
reported after the checks above are complete.

## Why Step 3 Uses Two Tables

### Two Different Roles

Step 3 separates the admissions we want to predict from the admission
history we search to determine what happened afterward.

| Table | Role | Question answered |
|---|---|---|
| Step 2 cohort (`cohort_step2.csv`) | Eligible index admissions | Which admissions receive a prediction and outcome label? |
| Source admissions (`admissions.csv`) | Candidate subsequent admission events | Did the same patient return after an index discharge? |

An **index admission** is the starting hospitalization for which we
predict readmission risk at discharge.

A **subsequent admission** is a later hospitalization that may count
as the outcome for an earlier index admission.

The same hospital stay can serve both roles, but eligibility for one
role does not automatically establish eligibility for the other.

### Example — A Return That Ends in Death

Consider this illustrative patient history:

| Stay | Admission date | Discharge date | Outcome of stay | Included in Step 2 cohort? |
|---|---|---|---|---|
| A | December 28 | January 1 | Discharged alive | Yes |
| B | January 10 | January 15 | Died during hospitalization | No |

For **Stay A**, the prediction question is:

> After discharge on January 1, did this patient return within 30 days?

The answer is **yes**: Stay B began on January 10, nine days after
Stay A's discharge. Assuming it meets the encounter-definition rules,
Stay B counts as a readmission for Stay A.

For **Stay B**, the prediction question would be:

> After this hospitalization ends, could the patient experience a
> subsequent readmission?

Because the patient died during Stay B, it is excluded as an index
admission under the Step 2 eligibility rules.

Therefore, Stay B is:
- **Excluded as a prediction observation.**
- **Retained as evidence of a return after Stay A.**

Death during Stay B does not erase the fact that the patient returned.

### What Goes Wrong If We Use Only the Step 2 Cohort?

The Step 2 cohort contains Stay A but excludes Stay B.

If we search only that filtered cohort, we cannot find Stay B.
The code may incorrectly assign Stay A a negative readmission label (Y=0),
even though a return was recorded in the source data.

Filtering the starting observations before searching for outcomes can
therefore remove information needed to label the remaining observations.

### How We Use the Two Tables

1. Load the Step 2 cohort as the set of eligible index admissions.
2. Load the source admission history as the pool of candidate return events.
3. Match events to the same patient using `subject_id`.
4. Use distinct admission identifiers and timestamps to determine whether
   a qualifying return occurred within 30 days after each index discharge.
5. Attach the resulting label to the index admission using `hadm_id`.

Loading the source history does not restore excluded admissions to the
prediction cohort. It supplies the events needed to determine outcomes.

Candidate return events still require validation. Overlaps, simultaneous
encounters, invalid timestamps, and transfers must be handled according
to the outcome definition; a source record does not automatically count
as a readmission.

### Key Principle

**Index-admission eligibility determines which stays we predict for.
Outcome-event eligibility determines which later stays count as returns.**

We use two tables because these are different decisions.

## 3.1 — Setup and Input Files

We identify two inputs:

- `cohort_step2.csv`: eligible index admissions that will receive labels.
- `admissions.csv`: source admission history used to find candidate returns.

We use the shared project configuration so paths remain consistent
across notebooks.

This section checks that both input files exist. It does not load,
filter, or label any admissions.

In [ ]:
from pathlib import Path
import pandas as pd

# Set this to your authorized local MIMIC-IV hospital data directory.
DATA_DIR = Path("data/raw")
ADMISSIONS_FILE = DATA_DIR / "admissions.csv"
PATIENTS_FILE = DATA_DIR / "patients.csv"
PROCESSED_DIR = Path("sample_outputs")
COHORT_FILE = PROCESSED_DIR / "cohort_step2.csv"

# Fail clearly when required local inputs are absent.
for input_path in [ADMISSIONS_FILE, PATIENTS_FILE, COHORT_FILE]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Required local input missing: {input_path}")


## 3.2 — Load and Validate Both Admission Tables

We load the eligible index cohort and the source admission history
as separate DataFrames.

Before examining admission sequences, we verify:
- Required columns are present.
- Patient and admission identifiers are not missing.
- Admission identifiers are unique within each table.
- Each index admission belongs to the same patient in the source history.

These checks prevent incorrect matching or duplicated observations.
No admissions are removed and no labels are assigned.

In [ ]:
# Load eligible starting admissions from Step 2.
index_cohort = pd.read_csv(COHORT_FILE)

# Load the full source history, including admissions excluded as index stays.
admission_history = pd.read_csv(ADMISSIONS_FILE)

# Confirm that both tables contain the fields needed for sequence review.
required_columns = {
    "subject_id", "hadm_id", "admittime", "dischtime",
    "admission_type", "admission_location", "discharge_location",
    "hospital_expire_flag", "deathtime"
}

for table_name, table in [
    ("Index cohort", index_cohort),
    ("Admission history", admission_history)
]:
    missing_columns = required_columns - set(table.columns)
    if missing_columns:
        raise ValueError(
            f"{table_name}: missing columns {sorted(missing_columns)}"
        )

    # Each admission needs both identifiers for reliable matching.
    if table[["subject_id", "hadm_id"]].isna().any().any():
        raise ValueError(f"{table_name}: missing identifiers.")

    # A repeated admission ID could duplicate records during later merges.
    if not table["hadm_id"].is_unique:
        raise ValueError(f"{table_name}: duplicate admission IDs.")

# Match each index admission to its source record.
# Compare patient/admission pairs, not just admission IDs.
index_source_check = index_cohort[["subject_id", "hadm_id"]].merge(
    admission_history[["subject_id", "hadm_id"]],
    on=["subject_id", "hadm_id"],
    how="left",
    validate="one_to_one",
    indicator=True
)

unmatched_count = int(
    index_source_check["_merge"].eq("left_only").sum()
)

if unmatched_count:
    raise ValueError(
        f"{unmatched_count} index admissions lack a matching source record."
    )

# Display aggregate counts without showing individual patient records.
input_summary = pd.DataFrame({
    "table": ["Index cohort", "Source admission history"],
    "admissions": [len(index_cohort), len(admission_history)],
    "unique_patients": [
        index_cohort["subject_id"].nunique(),
        admission_history["subject_id"].nunique()
    ]
})

display(input_summary)
print("Index admissions without a matching source record:", unmatched_count)
print("All identifier checks passed.")

,table,admissions,unique_patients
0,Index cohort,528551,216762
1,Source admission history,546028,223452


Index admissions without a matching source record: 0
All identifier checks passed.


### Interpretation — Input Validation

The index cohort contained 528,551 admissions from 216,762 patients.
The source history contained 546,028 admissions from 223,452 patients.

Both tables had complete identifiers and unique admission IDs.
Every index admission matched its patient/admission pair in the source
history. No records were removed during these checks.

## 3.3 — Timestamp Validation

We convert admission, discharge, and death timestamps in both tables
to datetime values.

We distinguish missing source values from values that fail to parse.
Missing death timestamps are reported separately because a death time
is not expected for every admission.

We then count admissions with missing admission/discharge timestamps,
reversed timestamps, or equal timestamps.

The source history includes records excluded from the index cohort,
so the two tables may have different results.

This diagnostic does not remove admissions or construct labels.

In [ ]:
timestamp_checks = []
time_order_checks = []

for table_name, table in [
    ("Index cohort", index_cohort),
    ("Source admission history", admission_history)
]:
    for column in ["admittime", "dischtime", "deathtime"]:
        # Read the current values before converting them.
        source_values = table[column]
        parsed_values = pd.to_datetime(source_values, errors="coerce")

        # Count missing values separately from failed conversions.
        timestamp_checks.append({
            "table": table_name,
            "column": column,
            "source_missing": int(source_values.isna().sum()),
            "parsing_failures": int(
                (source_values.notna() & parsed_values.isna()).sum()
            )
        })

        # Store datetime values for subsequent comparisons.
        table[column] = parsed_values

    # Only compare admission/discharge times when both are available.
    usable = table["admittime"].notna() & table["dischtime"].notna()

    time_order_checks.append({
        "table": table_name,
        "missing_admission_or_discharge": int((~usable).sum()),
        "discharge_before_admission": int(
            (usable & table["dischtime"].lt(table["admittime"])).sum()
        ),
        "equal_admission_discharge": int(
            (usable & table["dischtime"].eq(table["admittime"])).sum()
        )
    })

display(pd.DataFrame(timestamp_checks))
display(pd.DataFrame(time_order_checks))

,table,column,source_missing,parsing_failures
0,Index cohort,admittime,0,0
1,Index cohort,dischtime,0,0
2,Index cohort,deathtime,528551,0
3,Source admission history,admittime,0,0
4,Source admission history,dischtime,0,0
5,Source admission history,deathtime,534238,0


,table,missing_admission_or_discharge,discharge_before_admission,equal_admission_discharge
0,Index cohort,0,0,5
1,Source admission history,0,175,5


### Interpretation — Timestamp Validation

Admission and discharge timestamps were complete and parsed successfully
in both tables. No nonmissing death timestamps failed to parse.

The index cohort contained no reversed admission/discharge timestamps.
The source history retained the 175 reversed-time records identified
in Step 2. Both tables contained five equal-time stays.

All 528,551 index admissions had missing `deathtime`. This field does
not establish whether patients survived after discharge.

No records were removed during this diagnostic. Source records with
reversed timestamps require an explicit outcome-event decision before
labels are finalized.

## 3.4 — Examine Consecutive Admission Timing

We order source admissions within each patient and compare each discharge
with the next recorded admission.

A negative interval indicates overlapping records; a zero interval
indicates simultaneous discharge and admission. Neither automatically
establishes a distinct readmission.

This is a sequence diagnostic, not the final labeling rule.
No records are removed and no labels are assigned.

### Why We Validate the Timing of the Next Admission

A patient's next admission record is not automatically a readmission
after discharge. We first check whether it begins before, exactly at,
or after the index admission's discharge.

We calculate:

$$
\text{gap} =
\text{next recorded admission time}
-
\text{index discharge time}
$$

| Gap | Meaning |
|---|---|
| Negative | The next encounter begins before the index stay ends |
| Zero | The next encounter begins exactly when the index stay ends |
| Positive | The next encounter begins after index discharge |

### Example — Overlapping Admission Records

Consider this illustrative patient history:

| Stay | Admission time | Discharge time |
|---|---|---|
| A | January 1, 10 AM | January 5, 2 PM |
| B | January 5, 12 PM | January 8, 10 AM |

For Stay A:

$$
\text{gap} =
\text{January 5 at 12 PM}
-
\text{January 5 at 2 PM}
=
-2\text{ hours}
$$

Stay B begins two hours before Stay A's recorded discharge.
The two recorded hospitalization intervals therefore overlap.

This does not prove that the patient physically occupied two hospital
beds at once. Possible explanations include an administrative transition
between encounters, an incorrect timestamp, or duplicate encounter
records. The timing check alone cannot establish the cause.

### Difference from the Step 2 Timestamp Check

- **Step 2 checked within each admission:** Did discharge occur before
  that admission began?
- **Step 3 checks between admissions:** Did another encounter begin
  before the index admission ended?

Two stays can each have valid admission/discharge ordering while
overlapping one another.

### Why This Matters for Label Construction

An overlapping encounter cannot automatically be treated as a distinct
return after discharge. However, assigning a negative label solely
because the next recorded gap is negative could also be incorrect.

For example, an overlapping Stay B may be followed by a separate Stay C
that genuinely begins within 30 days after Stay A's discharge.
We must not overlook Stay C merely because Stay B was the next row.

A zero-gap transition also requires review: it may represent a separate
encounter or continuation of care. Equal timestamps alone do not
establish which interpretation is correct.


### Length of Stay versus Time Between Admissions

These calculations answer different questions:

| Calculation | Meaning |
|---|---|
| Stay A discharge − Stay A admission | How long did Stay A last? |
| Stay B admission − Stay A discharge | How long after Stay A ended did Stay B begin? |

For example:

| Stay | Admission time | Discharge time |
|---|---|---|
| A | January 1, 10 AM | January 5, 2 PM |
| B | January 5, 12 PM | January 8, 10 AM |

**Length of Stay A:**

January 5, 2 PM − January 1, 10 AM = **4 days, 4 hours**

This measures time spent in the index hospitalization.

**Gap from Stay A to Stay B:**

January 5, 12 PM − January 5, 2 PM = **−2 hours**

This measures the timing of the next encounter relative to index discharge.

### Why Step 3 Uses the Second Calculation

Our prediction point is discharge, so the 30-day outcome window begins
when Stay A ends—not when Stay A begins.

We therefore calculate:

$$
\text{Time to return}
=
\text{subsequent admission time}
-
\text{index discharge time}
$$

In this example, Stay B begins before Stay A's recorded discharge.
Although Stay A has a valid positive length of stay, the two records
overlap. Stay B cannot automatically be treated as a distinct return
after discharge.

### Why an Overlapping Stay Is Not Automatically a Readmission

A readmission after discharge requires the index stay to have ended
before the qualifying return begins.

In our example:

- Stay B begins January 5 at **12 PM**.
- Stay A ends January 5 at **2 PM**.

At B's recorded admission time, A has not yet ended. The interval is:

$$
\text{Stay B admission} - \text{Stay A discharge}
= -2\text{ hours}
$$

The records therefore overlap. We cannot establish from these timestamps
that the patient completed Stay A and then returned for Stay B.

Possible explanations include a new administrative encounter during
continuing care or an incorrectly recorded timestamp. These are
possibilities, not confirmed explanations.

A different `hadm_id` identifies a separate admission record, but does
not by itself prove a distinct return after discharge.

### Implication for Label Construction

We do not automatically count overlapping Stay B as a readmission.
We also do not automatically assign Stay A a negative label.

A later Stay C could begin after A's discharge and within its 30-day
window. We must consider qualifying subsequent events rather than rely
only on the next row in admission order.

**Decision:** Flag overlapping encounters for investigation and apply
documented encounter rules before finalizing readmission labels.

This is why we validate the interval between encounters before assigning
readmission labels. We must distinguish overlapping or simultaneous
records from qualifying returns within the 30-day window.

**Key reminder:** Length of stay measures time inside the index stay.
Time to return measures the interval from index discharge to a later admission.

In [ ]:
# Create a separate ordered table for sequence review.
history_sequence = admission_history.sort_values(
    ["subject_id", "admittime", "hadm_id"]
).reset_index(drop=True).copy()

# Get the next recorded admission within each patient.
# hadm_id breaks timestamp ties for reproducibility, not clinical ordering.
history_sequence["next_recorded_admittime"] = (
    history_sequence.groupby("subject_id")["admittime"].shift(-1)
)

# Preserve the exact elapsed interval rather than truncate to whole days.
history_sequence["gap_to_next_recorded"] = (
    history_sequence["next_recorded_admittime"]
    - history_sequence["dischtime"]
)

# Limit the summary to eligible index admissions.
index_sequence_review = history_sequence.loc[
    history_sequence["hadm_id"].isin(index_cohort["hadm_id"])
].copy()

gap = index_sequence_review["gap_to_next_recorded"]
has_next = index_sequence_review["next_recorded_admittime"].notna()
zero = pd.Timedelta(0)
one_day = pd.Timedelta(days=1)
thirty_days = pd.Timedelta(days=30)

# These categories are mutually exclusive.
sequence_summary = pd.DataFrame({
    "category": [
        "No subsequent source admission recorded",
        "Next admission begins before index discharge",
        "Next admission begins exactly at index discharge",
        "Next admission begins >0 to 24 hours after discharge",
        "Next admission begins >24 hours to 30 days after discharge",
        "Next admission begins more than 30 days after discharge"
    ],
    "admissions": [
        int((~has_next).sum()),
        int((has_next & gap.lt(zero)).sum()),
        int((has_next & gap.eq(zero)).sum()),
        int((has_next & gap.gt(zero) & gap.le(one_day)).sum()),
        int((has_next & gap.gt(one_day) & gap.le(thirty_days)).sum()),
        int((has_next & gap.gt(thirty_days)).sum())
    ]
})

assert sequence_summary["admissions"].sum() == len(index_cohort), \
    "Sequence categories do not reconcile with the index cohort."

display(sequence_summary)

,category,admissions
0,No subsequent source admission recorded,207015
1,Next admission begins before index discharge,47
2,Next admission begins exactly at index discharge,2068
3,Next admission begins >0 to 24 hours after dis...,11522
4,Next admission begins >24 hours to 30 days aft...,94939
5,Next admission begins more than 30 days after ...,212960


### Findings from the Initial Sequence Check

Among 528,551 eligible index admissions:

- **47** had a next recorded admission beginning before index discharge.
- **2,068** had a next recorded admission beginning exactly at discharge.

These counts describe index admissions, not necessarily unique patients.
They identify records for further investigation, not confirmed errors.

### Current Decision

This section is diagnostic only. No records are excluded and no
readmission labels are assigned.

We will review these transitions and document the encounter rules before
selecting qualifying return events.

**Key reminder:** The next row in admission order is not necessarily
the next qualifying readmission after discharge.

## 3.5 — Investigate Overlapping and Simultaneous Encounters

We examine index admissions whose next recorded admission begins
before or exactly at index discharge.

We attach the next encounter's admission type, admission location,
and timestamp information to understand the recorded transition.

We summarize:
- The size of negative intervals.
- Index discharge destinations and next admission sources.
- Admission types of the next encounters.
- Whether the next encounter itself has reversed timestamps.

These patterns may help explain administrative transitions, but do not
prove that a transition is a transfer, duplicate, or separate readmission.

No admissions are removed and no labels are assigned.

In [ ]:
# Work on a separate copy so the existing tables remain unchanged.
transition_review = history_sequence.copy()

# Attach details from the next admission within the same patient.
# These shifts use the same ordering established in Section 3.4.
next_columns = [
    "hadm_id",
    "dischtime",
    "admission_type",
    "admission_location"
]

for column in next_columns:
    transition_review[f"next_{column}"] = (
        transition_review.groupby("subject_id")[column].shift(-1)
    )

# Select eligible index admissions with a next encounter at or before discharge.
gap = transition_review["gap_to_next_recorded"]

transition_review = transition_review.loc[
    transition_review["hadm_id"].isin(index_cohort["hadm_id"])
    & gap.notna()
    & gap.le(pd.Timedelta(0))
].copy()

# Separate negative gaps from exactly zero gaps.
transition_review["transition_type"] = "Overlap"
transition_review.loc[
    transition_review["gap_to_next_recorded"].eq(pd.Timedelta(0)),
    "transition_type"
] = "Zero gap"

# Express the gap in hours for interpretation; retain the original timedelta.
transition_review["gap_hours"] = (
    transition_review["gap_to_next_recorded"].dt.total_seconds() / 3600
)

print("Transitions reviewed:")
display(
    transition_review.groupby("transition_type")
    .agg(
        index_admissions=("hadm_id", "size"),
        unique_patients=("subject_id", "nunique")
    )
)

print("Negative gap duration in hours:")
print(
    transition_review.loc[
        transition_review["transition_type"].eq("Overlap"),
        "gap_hours"
    ].describe()
)

# Summarize recorded destinations and sources, including missing values.
print("Index discharge location and next admission location:")
display(
    transition_review.groupby(
        ["transition_type", "discharge_location", "next_admission_location"],
        dropna=False
    )
    .size()
    .rename("transitions")
    .reset_index()
    .sort_values(
        ["transition_type", "transitions"],
        ascending=[True, False]
    )
    .groupby("transition_type", sort=False)
    .head(10)
)

print("Next encounter admission types:")
display(
    transition_review.groupby(
        ["transition_type", "next_admission_type"],
        dropna=False
    )
    .size()
    .rename("transitions")
    .reset_index()
)

# Check whether the next encounter has its own reversed timestamps.
transition_review["next_stay_reversed"] = (
    transition_review["next_dischtime"]
    < transition_review["next_recorded_admittime"]
)

print("Next encounters with reversed admission/discharge timestamps:")
display(
    transition_review.groupby("transition_type")["next_stay_reversed"]
    .sum()
    .to_frame("transitions")
)

Transitions reviewed:


,index_admissions,unique_patients
transition_type,,
Overlap,47,47
Zero gap,2068,1850


Negative gap duration in hours:
count     47.000000
mean     -50.180142
std      149.100087
min     -743.700000
25%      -23.241667
50%      -13.883333
75%       -1.008333
max       -0.033333
Name: gap_hours, dtype: float64
Index discharge location and next admission location:


,transition_type,discharge_location,next_admission_location,transitions
11,Overlap,NaN,EMERGENCY ROOM,16
12,Overlap,NaN,PHYSICIAN REFERRAL,6
2,Overlap,HOME,EMERGENCY ROOM,5
3,Overlap,HOME,PHYSICIAN REFERRAL,4
6,Overlap,PSYCH FACILITY,PHYSICIAN REFERRAL,3
5,Overlap,PSYCH FACILITY,INTERNAL TRANSFER TO OR FROM PSYCH,2
7,Overlap,REHAB,EMERGENCY ROOM,2
8,Overlap,SKILLED NURSING FACILITY,EMERGENCY ROOM,2
13,Overlap,NaN,WALK-IN/SELF REFERRAL,2
0,Overlap,ACUTE HOSPITAL,PHYSICIAN REFERRAL,1


Next encounter admission types:


,transition_type,next_admission_type,transitions
0,Overlap,DIRECT EMER.,10
1,Overlap,DIRECT OBSERVATION,1
2,Overlap,EU OBSERVATION,20
3,Overlap,EW EMER.,9
4,Overlap,OBSERVATION ADMIT,1
5,Overlap,SURGICAL SAME DAY ADMISSION,1
6,Overlap,URGENT,5
7,Zero gap,DIRECT EMER.,385
8,Zero gap,DIRECT OBSERVATION,22
9,Zero gap,ELECTIVE,2


Next encounters with reversed admission/discharge timestamps:


,transitions
transition_type,
Overlap,0
Zero gap,0


### Interpretation of overlapping and zero-gap admission records

This check examines the timing between each eligible index admission
and the next recorded admission for the same patient in the full
admission history.

The interval is calculated as:

**Gap = next admission time − current discharge time**

#### Findings

- **47 overlapping pairs across 47 patients:** the next admission
  begins before the current admission ends. The median gap is
  approximately −13.9 hours, with overlaps ranging from 2 minutes
  to approximately 31 days.

- **2,068 zero-gap pairs across 1,850 patients:** the next admission
  begins exactly when the current admission ends. Some patients
  contribute more than one such pair.

- Several zero-gap pairs involve psychiatric facilities or an
  admission location recorded as `INTERNAL TRANSFER TO OR FROM PSYCH`.
  These patterns suggest that some pairs may represent continuing
  care or administrative transitions rather than a distinct return
  after discharge. The available fields do not confirm the cause.

- None of the next admissions in these reviewed pairs have a discharge
  time earlier than their own admission time. Therefore, internally
  valid stay timestamps do not guarantee clear timing between stays.

#### Implications for readmission labeling

A different `hadm_id` identifies a separate admission record, but does
not by itself establish a distinct readmission after discharge.

An overlapping admission starts before the readmission window begins.
A zero-gap admission occurs at its boundary and requires an explicit
rule about whether it qualifies as a separate return.

These findings do not justify automatically assigning either a positive
or a negative readmission label. A later admission may still represent
a qualifying return within 30 days.

At this stage, no admissions are removed and no final labels are
assigned. The next step is to inspect selected patient timelines and
document how ambiguous transitions will be handled.

**Scope:** this check examines consecutive records ordered by admission
time; it is not an exhaustive search for every pair of overlapping stays.

### 3.6 Inspect selected patient admission timelines

The previous check identified overlapping and zero-gap admission pairs.
Summary counts alone cannot show how these pairs relate to the patient's
other hospital stays.

We inspect:
- The three pairs with the largest overlaps.
- Two zero-gap pairs involving an internal psychiatric transfer.
- Two other zero-gap pairs.

For each selected patient, we display the full recorded admission history,
including admissions excluded from the Step 2 index cohort.

This review helps us examine whether the records suggest continuing care,
an administrative transition, or a separate return. These examples cannot
establish the cause of every ambiguous transition.

No exclusions or final readmission labels are applied in this section.

In [ ]:
# Select examples without hard-coding any patient or admission IDs.
overlap_examples = (
    transition_review[
        transition_review["transition_type"].eq("Overlap")
    ]
    .sort_values("gap_hours")
    .head(3)
)

is_zero_gap = transition_review["transition_type"].eq("Zero gap")

is_psych_transfer = (
    transition_review["next_admission_location"]
    .eq("INTERNAL TRANSFER TO OR FROM PSYCH")
    .fillna(False)
)

psych_examples = transition_review[
    is_zero_gap & is_psych_transfer
].head(2)

other_zero_examples = transition_review[
    is_zero_gap & ~is_psych_transfer
].head(2)

selected_pairs = pd.concat(
    [overlap_examples, psych_examples, other_zero_examples]
).drop_duplicates(subset=["hadm_id"])


# Display aliases and relative times instead of actual IDs and dates.
for patient_number, (subject_id, pairs) in enumerate(
    selected_pairs.groupby("subject_id", sort=False),
    start=1,
):
    timeline = (
        admission_history[
            admission_history["subject_id"].eq(subject_id)
        ]
        .sort_values(["admittime", "hadm_id"])
        .copy()
    )

    # Give each admission a readable alias within this patient's history.
    stay_aliases = {
        hadm_id: f"Stay_{number:02d}"
        for number, hadm_id in enumerate(timeline["hadm_id"], start=1)
    }

    # Use relative hours to preserve timing without displaying dates.
    # Hour 0 is this patient's earliest recorded admission.
    time_origin = timeline["admittime"].min()

    timeline["admission_hour"] = (
        (timeline["admittime"] - time_origin)
        .dt.total_seconds()
        .div(3600)
        .round(2)
    )

    timeline["discharge_hour"] = (
        (timeline["dischtime"] - time_origin)
        .dt.total_seconds()
        .div(3600)
        .round(2)
    )

    timeline["stay_alias"] = timeline["hadm_id"].map(stay_aliases)

    timeline["in_step2_cohort"] = timeline["hadm_id"].isin(
        index_cohort["hadm_id"]
    )

    timeline["selected_index_A"] = timeline["hadm_id"].isin(
        pairs["hadm_id"]
    )

    timeline["selected_next_B"] = timeline["hadm_id"].isin(
        pairs["next_hadm_id"].dropna()
    )

    # Build a separate display table that contains aliases only.
    pair_display = pairs[
        ["transition_type", "gap_hours"]
    ].copy()

    pair_display.insert(
        0, "index_stay_A", pairs["hadm_id"].map(stay_aliases)
    )
    pair_display.insert(
        1, "next_stay_B", pairs["next_hadm_id"].map(stay_aliases)
    )

    print(f"\nPatient_{patient_number:02d}")
    print("Times are hours since this patient's first recorded admission.")

    display(pair_display.reset_index(drop=True))

    display(
        timeline[
            [
                "stay_alias",
                "admission_hour",
                "discharge_hour",
                "admission_type",
                "admission_location",
                "discharge_location",
                "hospital_expire_flag",
                "in_step2_cohort",
                "selected_index_A",
                "selected_next_B",
            ]
        ].reset_index(drop=True)
    )

### Findings from selected patient timelines

The selected examples show both partial overlaps and admissions entirely
contained within another admission's recorded time interval.

Some zero-gap pairs connect an observation encounter to an urgent
admission with an internal psychiatric transfer location. This suggests
possible continuing care or an administrative transition, but does not
confirm the explanation for each pair.

Zero-gap pairs also occur without an explicit internal-transfer location.
Therefore, admission location alone cannot resolve all ambiguous pairs.

The examples also show that one admission can overlap multiple subsequent
records. Checking only the immediately next admission is therefore not
an exhaustive assessment of overlapping stays.

These findings support defining an explicit approach to overlapping and
zero-gap records before constructing the outcome. They do not justify
automatically assigning a negative label, because another later admission
may still qualify within the 30-day window.

No exclusions or final labels have been applied in this review.

### 3.7 Identify admissions connected by overlapping or touching intervals

The timeline review showed that one long admission can overlap several
shorter admissions. Checking only the immediately next record does not
identify every admission involved.

For each admission, we check:

1. Does it begin before or exactly when an earlier admission ends?
2. Does the next admission begin before or exactly when this admission ends?

For the first check, we use the latest discharge time among all earlier
admissions for the same patient. This detects admissions nested within
an earlier, longer stay.

Records with discharge before admission are excluded from this interval
check because their intervals are invalid. They remain in the source table
for separate review.

These flags describe ambiguous timing connections, not confirmed transfers
or data errors. This step counts affected Step 2 admissions without removing
them or assigning readmission labels.

In [ ]:
# Use valid intervals for this diagnostic.
# Keep the original admission_history unchanged.
interval_review = (
    admission_history.loc[
        admission_history["admittime"].notna()
        & admission_history["dischtime"].notna()
        & admission_history["dischtime"].ge(
            admission_history["admittime"]
        )
    ]
    .sort_values(["subject_id", "admittime", "hadm_id"])
    .reset_index(drop=True)
    .copy()
)

# Track the latest discharge time reached so far for each patient.
# This accounts for long stays containing several shorter records.
interval_review["latest_discharge_so_far"] = (
    interval_review.groupby("subject_id")["dischtime"].cummax()
)

# Shift within each patient to exclude the current admission itself.
interval_review["latest_prior_discharge"] = (
    interval_review.groupby("subject_id")[
        "latest_discharge_so_far"
    ].shift(1)
)

# Find the next admission start within the valid-interval history.
interval_review["next_valid_admittime"] = (
    interval_review.groupby("subject_id")["admittime"].shift(-1)
)

# Incoming connection:
# this stay begins before or exactly when an earlier stay ends.
interval_review["connected_to_prior"] = (
    interval_review["admittime"]
    .le(interval_review["latest_prior_discharge"])
)

# Outgoing connection:
# another stay begins before or exactly when this stay ends.
interval_review["connected_to_next"] = (
    interval_review["next_valid_admittime"]
    .le(interval_review["dischtime"])
)

# Flag either side of an overlapping or touching connection.
interval_review["timing_connection"] = (
    interval_review["connected_to_prior"]
    | interval_review["connected_to_next"]
)

# Summarize only admissions eligible as Step 2 index observations.
index_interval_review = interval_review.loc[
    interval_review["hadm_id"].isin(index_cohort["hadm_id"])
].copy()

# Confirm every Step 2 admission was included exactly once.
assert len(index_interval_review) == len(index_cohort)
assert index_interval_review["hadm_id"].is_unique

summary = pd.DataFrame({
    "timing_status": [
        "No overlapping or touching connection",
        "Overlapping or touching connection",
    ],
    "index_admissions": [
        (~index_interval_review["timing_connection"]).sum(),
        index_interval_review["timing_connection"].sum(),
    ],
})

summary["percent_of_step2"] = (
    summary["index_admissions"]
    .div(len(index_cohort))
    .mul(100)
    .round(2)
)

display(summary)

print(
    "Unique patients with affected index admissions:",
    index_interval_review.loc[
        index_interval_review["timing_connection"],
        "subject_id",
    ].nunique()
)

,timing_status,index_admissions,percent_of_step2
0,No overlapping or touching connection,524357,99.21
1,Overlapping or touching connection,4194,0.79


Unique patients with affected index admissions: 1891


### Interpretation

Among the 528,551 Step 2 index admissions:

- 524,357 (99.21%) have no overlapping or touching connection
  with another valid admission interval for the same patient.
- 4,194 (0.79%) have at least one such connection.
- The affected index admissions involve 1,891 unique patients.

These counts describe individual index admissions, whereas the earlier
diagnostic counted consecutive admission pairs. The totals therefore
are not directly comparable.

The flags identify ambiguous timing relationships; they do not establish
that the records are errors or confirmed transfers. Likewise, an unflagged
admission is not automatically guaranteed a valid readmission label:
outcome eligibility and follow-up still require assessment.

No admissions have been removed, and no final labels have been assigned.tsen

### 3.8 Handle admissions with ambiguous timing connections

For the primary analysis, we exclude Step 2 index admissions that overlap
or touch another valid admission interval for the same patient.

This is a conservative methodological choice. These records are not
necessarily incorrect, but their timing does not provide an unambiguous
starting point for a separate post-discharge prediction.

This rule excludes 4,194 index admissions (0.79%), leaving 524,357
potential index admissions before further outcome and follow-up checks.

The excluded records remain in the full admission history. Exclusion
as an index observation does not automatically disqualify an admission
from representing a return after an earlier discharge.

This restriction may change the population represented by the analysis.
An alternative approach, such as combining connected stays into episodes,
would require separate cohort, feature, and outcome definitions.

No final readmission labels are assigned in this section.

In [ ]:
# Identify admissions excluded as index observations.
ambiguous_index_ids = index_interval_review.loc[
    index_interval_review["timing_connection"],
    "hadm_id",
]

# Preserve Step 2 and the full admission history.
# Create a separate table for the primary labeling analysis.
index_for_labeling = index_cohort.loc[
    ~index_cohort["hadm_id"].isin(ambiguous_index_ids)
].copy()

# Verify that only the intended index admissions were excluded.
assert index_for_labeling["hadm_id"].is_unique
assert len(index_for_labeling) + len(ambiguous_index_ids) == len(index_cohort)
assert not index_for_labeling["hadm_id"].isin(ambiguous_index_ids).any()

print("Step 2 index admissions:", len(index_cohort))
print("Excluded for ambiguous timing:", len(ambiguous_index_ids))
print("Remaining potential index admissions:", len(index_for_labeling))
print(
    "Remaining unique patients:",
    index_for_labeling["subject_id"].nunique(),
)
print("Full admission history retained:", len(admission_history))

Step 2 index admissions: 528551
Excluded for ambiguous timing: 4194
Remaining potential index admissions: 524357
Remaining unique patients: 216046
Full admission history retained: 546028


### 3.9 Define candidate return admissions

Index eligibility and return-event eligibility serve different purposes.

An index admission provides the discharge from which we predict a future
return. A candidate return admission provides evidence that the patient
was admitted again.

#### Candidate return rules

For the primary analysis:

- Use the full source admissions table, rather than only the index cohort.
- Retain all admission types, including elective and observation admissions.
- Retain admissions ending in death or hospice.
- Retain admissions excluded as index observations for overlapping or
  touching intervals. Their relationship to an earlier index discharge
  must be assessed separately.
- Require nonmissing admission and discharge timestamps, with discharge
  at or after admission.
- Retain equal admission/discharge timestamps under this timing rule.

Records with missing or reversed timestamps are placed in a separate
review table. They are not used as confirmed return events in the primary
definition. However, we must check whether they fall within an index
admission's outcome window before assigning a negative label.

#### Relationship to an index admission

A candidate becomes a qualifying 30-day return only when it:

1. Belongs to the same patient.
2. Has a different admission ID.
3. Begins strictly after the index discharge and no more than 30 days later.

Thus, the time condition is:

**0 < return admission time − index discharge time ≤ 30 days**

Positive gaps shorter than 24 hours remain eligible. This analysis does
not impose a minimum one-day interval.

This target concerns recorded returns represented in admissions.csv,
including planned returns and observation admissions. It is not restricted
to unplanned inpatient readmission.

This section prepares candidate tables only. Final labels and follow-up
limitations are handled separately.

In [ ]:
# Assess timestamps in the full source admission history.
# Do not apply the Step 2 index exclusions to return candidates.
has_both_times = (
    admission_history["admittime"].notna()
    & admission_history["dischtime"].notna()
)

reversed_times = (
    has_both_times
    & admission_history["dischtime"].lt(
        admission_history["admittime"]
    )
)

valid_return_timing = has_both_times & ~reversed_times


# Candidate returns include all admission types and discharge outcomes.
return_candidates = admission_history.loc[
    valid_return_timing
].copy()

# Preserve questionable records for a later outcome-window check.
# Silently dropping these could produce false negative labels.
return_timing_review = admission_history.loc[
    ~valid_return_timing
].copy()

return_timing_review["timing_issue"] = (
    "Missing admission or discharge timestamp"
)

return_timing_review.loc[
    return_timing_review["dischtime"].lt(
        return_timing_review["admittime"]
    ),
    "timing_issue",
] = "Discharge before admission"


# Verify that every source admission is accounted for.
assert return_candidates["hadm_id"].is_unique
assert return_timing_review["hadm_id"].is_unique

assert (
    len(return_candidates) + len(return_timing_review)
    == len(admission_history)
)

assert return_candidates["dischtime"].ge(
    return_candidates["admittime"]
).all()


# Display aggregate counts only.
print("Full source admissions:", len(admission_history))
print("Candidate return admissions:", len(return_candidates))
print("Admissions requiring timing review:", len(return_timing_review))
print("Potential index admissions:", len(index_for_labeling))

print("\nTiming issues:")
display(
    return_timing_review["timing_issue"]
    .value_counts()
    .rename_axis("timing_issue")
    .to_frame("admissions")
)

print("\nCandidate return admission types:")
display(
    return_candidates["admission_type"]
    .value_counts(dropna=False)
    .rename_axis("admission_type")
    .to_frame("admissions")
)

Full source admissions: 546028
Candidate return admissions: 545853
Admissions requiring timing review: 175
Potential index admissions: 524357

Timing issues:


,admissions
timing_issue,
Discharge before admission,175



Candidate return admission types:


,admissions
admission_type,
EW EMER.,177381
EU OBSERVATION,119400
OBSERVATION ADMIT,84429
URGENT,54910
SURGICAL SAME DAY ADMISSION,42896
DIRECT OBSERVATION,24545
DIRECT EMER.,21968
ELECTIVE,13130
AMBULATORY OBSERVATION,7194


### Validation summary

The source history contains 546,028 admissions. Of these, 545,853 satisfy
the candidate-return timestamp rules, while 175 have discharge before
admission and remain in a separate timing-review table.

All nine admission types remain represented among return candidates.
These counts describe potential return records, not observed readmissions.

The index table contains 524,357 admissions. The candidate-return table
is larger because index exclusions are not automatically applied to
return events.

Next, we will find the earliest candidate admission strictly after each
index discharge and calculate the exact interval. We will also check
whether questionable records occur within the 30-day window before
assigning negative labels.

### 3.10 Match index discharges to the next candidate admission

For each index admission, we find the same patient's earliest candidate
admission starting strictly after the index discharge.

We then calculate:

**Gap = candidate admission time − index discharge time**

A recorded return within 30 days satisfies:

**0 < gap ≤ 30 days**

We compare exact time intervals rather than rounded whole days.
For example, 30 days and 12 hours is outside the window.

If no later candidate admission is found, the return timestamp and gap
remain missing. Missing gaps are not replaced with artificial values.

This step identifies observed return timing only. Final negative labels
remain pending checks of questionable records and follow-up limitations.

All matched future-admission fields are outcome-audit information and
must not be used as model predictors.

In [ ]:
# Prepare candidate return fields.
# Renaming prevents confusion with the index admission's own fields.
candidate_lookup = return_candidates[
    ["subject_id", "hadm_id", "admittime", "admission_type"]
].rename(
    columns={
        "hadm_id": "return_hadm_id",
        "admittime": "return_admittime",
        "admission_type": "return_admission_type",
    }
).copy()

# Nullable integer preserves admission IDs when a match is missing.
candidate_lookup["return_hadm_id"] = (
    candidate_lookup["return_hadm_id"].astype("Int64")
)

# merge_asof requires the matching time columns to be globally sorted.
index_sorted = index_for_labeling.sort_values(
    ["dischtime", "subject_id", "hadm_id"]
)

candidate_sorted = candidate_lookup.sort_values(
    ["return_admittime", "subject_id", "return_hadm_id"]
)

# Find the earliest candidate admission AFTER each index discharge.
# by: match only within the same patient.
# direction="forward": search forward from discharge.
# allow_exact_matches=False: exclude zero-gap matches.
return_matches = pd.merge_asof(
    index_sorted,
    candidate_sorted,
    left_on="dischtime",
    right_on="return_admittime",
    by="subject_id",
    direction="forward",
    allow_exact_matches=False,
)

# If multiple candidates share the same admission timestamp,
# sorting by admission ID provides reproducible selection.
# Their time gap is identical; the ID order has no clinical meaning.

# Calculate the exact interval, preserving missing values.
return_matches["gap_to_return"] = (
    return_matches["return_admittime"]
    - return_matches["dischtime"]
)

# This flag identifies observed positives only.
# False does NOT yet constitute a finalized negative label.
return_matches["observed_return_within_30d"] = (
    return_matches["gap_to_return"].gt(pd.Timedelta(0))
    & return_matches["gap_to_return"].le(pd.Timedelta(days=30))
)


# Validate the matching.
assert len(return_matches) == len(index_for_labeling)
assert return_matches["hadm_id"].is_unique

has_match = return_matches["return_admittime"].notna()

assert return_matches.loc[
    has_match, "gap_to_return"
].gt(pd.Timedelta(0)).all()

assert (
    return_matches.loc[has_match, "hadm_id"]
    != return_matches.loc[has_match, "return_hadm_id"]
).all()


# Summarize timing without displaying patient-level records.
timing_summary = pd.DataFrame({
    "return_timing": [
        "Candidate return within 30 days",
        "Next candidate return after 30 days",
        "No later candidate return found",
    ],
    "index_admissions": [
        return_matches["observed_return_within_30d"].sum(),
        return_matches["gap_to_return"].gt(
            pd.Timedelta(days=30)
        ).sum(),
        (~has_match).sum(),
    ],
})

assert (
    timing_summary["index_admissions"].sum()
    == len(index_for_labeling)
)

display(timing_summary)

,return_timing,index_admissions
0,Candidate return within 30 days,106115
1,Next candidate return after 30 days,212175
2,No later candidate return found,206067


### Interpretation of return matching

Matching identified a candidate return within 30 days for 106,115 index
admissions. For 212,175 admissions, the next candidate return occurred
after 30 days. For 206,067 admissions, no later candidate return was found.

The three groups account for all 524,357 potential index admissions.

The latter two groups both lack an observed qualifying return within
30 days, but they preserve different information: a known later return
versus no later candidate return recorded.

These are matching results, not final modeling label counts.
Before finalizing negative labels, we will assess whether the 175
invalid-timing records create outcome uncertainty and document the
follow-up assumptions.

### 3.11 Check invalid-timing records within outcome windows

The 175 records with discharge before admission were excluded from
confirmed return candidates. However, their recorded admission times
may suggest a return after an earlier, valid index discharge.

For each index admission, we check whether a questionable record for
the same patient has a recorded admission time satisfying:

**0 < questionable admission time − index discharge time ≤ 30 days**

We distinguish:

- A valid return within 30 days: positive evidence remains available,
  whether or not a questionable record also appears.
- No valid return within 30 days, but a questionable record inside
  the window: flag the outcome as uncertain pending review.
- No valid return and no questionable record inside the window:
  no timing-related uncertainty detected by this check; follow-up
  assessment is still required.

This check uses recorded admission times without correcting them.
It cannot establish the true timing of an invalid record.

No admissions are removed and no final negative labels are assigned
in this section.

In [ ]:
# This implementation requires recorded admission times for all
# questionable records. Stop if any cannot be located in time.
assert return_timing_review["admittime"].notna().all(), (
    "Some questionable records lack admission times and need "
    "a separate review before this check."
)

# Prepare questionable admission timestamps.
questionable_lookup = (
    return_timing_review[
        ["subject_id", "hadm_id", "admittime"]
    ]
    .rename(columns={
        "hadm_id": "questionable_hadm_id",
        "admittime": "questionable_admittime",
    })
    .copy()
)

questionable_lookup["questionable_hadm_id"] = (
    questionable_lookup["questionable_hadm_id"].astype("Int64")
)

# Find the earliest questionable admission strictly after discharge.
# If the earliest is outside 30 days, all later ones are outside too.
timing_audit = pd.merge_asof(
    return_matches.sort_values(
        ["dischtime", "subject_id", "hadm_id"]
    ),
    questionable_lookup.sort_values(
        ["questionable_admittime", "subject_id", "questionable_hadm_id"]
    ),
    left_on="dischtime",
    right_on="questionable_admittime",
    by="subject_id",
    direction="forward",
    allow_exact_matches=False,
)

timing_audit["gap_to_questionable"] = (
    timing_audit["questionable_admittime"]
    - timing_audit["dischtime"]
)

timing_audit["questionable_record_within_30d"] = (
    timing_audit["gap_to_questionable"].gt(pd.Timedelta(0))
    & timing_audit["gap_to_questionable"].le(
        pd.Timedelta(days=30)
    )
)

# Flag cases where a questionable record could change an apparent
# negative into a positive if its timing were validated.
timing_audit["timing_outcome_uncertain"] = (
    ~timing_audit["observed_return_within_30d"]
    & timing_audit["questionable_record_within_30d"]
)

# Confirm one row remains for every potential index admission.
assert len(timing_audit) == len(return_matches)
assert timing_audit["hadm_id"].is_unique

# Show all four combinations, including any zero-count groups.
audit_summary = (
    pd.crosstab(
        timing_audit["observed_return_within_30d"],
        timing_audit["questionable_record_within_30d"],
    )
    .reindex(
        index=[False, True],
        columns=[False, True],
        fill_value=0,
    )
    .rename_axis(
        index="Valid return within 30 days",
        columns="Questionable record within 30 days",
    )
)

display(audit_summary)

print(
    "Index admissions with timing-related outcome uncertainty:",
    timing_audit["timing_outcome_uncertain"].sum(),
)

print(
    "Unique patients with timing-related outcome uncertainty:",
    timing_audit.loc[
        timing_audit["timing_outcome_uncertain"],
        "subject_id",
    ].nunique(),
)

Questionable record within 30 days,False,True
Valid return within 30 days,,
False,418219,23
True,106101,14


Index admissions with timing-related outcome uncertainty: 23
Unique patients with timing-related outcome uncertainty: 23


In [ ]:
# Select index admissions with a questionable record inside 30 days.
affected_37 = timing_audit.loc[
    timing_audit["questionable_record_within_30d"]
].copy()

# Show up to three examples from each group.
examples = pd.concat([
    affected_37.loc[
        affected_37["timing_outcome_uncertain"]
    ].head(3),

    affected_37.loc[
        ~affected_37["timing_outcome_uncertain"]
    ].head(3),
])

# Look up the questionable record's discharge timestamp.
questionable_details = return_timing_review[
    ["hadm_id", "dischtime"]
].rename(columns={
    "hadm_id": "questionable_hadm_id",
    "dischtime": "questionable_dischtime",
})

examples = examples.merge(
    questionable_details,
    on="questionable_hadm_id",
    how="left",
    validate="many_to_one",
)

for number, (_, row) in enumerate(examples.iterrows(), start=1):

    # Hour 0 = index discharge.
    # Negative hours occur before discharge; positive hours occur after.
    origin = row["dischtime"]

    def hours_from_discharge(timestamp):
        if pd.isna(timestamp):
            return float("nan")
        return round(
            (timestamp - origin).total_seconds() / 3600,
            2,
        )

    sample = pd.DataFrame([
        {
            "record": "Index stay A",
            "admission_hour": hours_from_discharge(row["admittime"]),
            "discharge_hour": 0.0,
        },
        {
            "record": "Questionable stay B",
            "admission_hour": hours_from_discharge(
                row["questionable_admittime"]
            ),
            "discharge_hour": hours_from_discharge(
                row["questionable_dischtime"]
            ),
        },
    ])

    print(f"\nExample_{number:02d}")
    print("Hour 0 = index discharge; 30 days = 720 hours.")
    display(sample)

    # Report the valid candidate separately.
    if pd.notna(row["return_admittime"]):
        print(
            "Earliest valid candidate return:",
            hours_from_discharge(row["return_admittime"]),
            "hours after index discharge",
        )
    else:
        print("No later valid candidate return found.")

    print(
        "Valid return within 30 days:",
        bool(row["observed_return_within_30d"]),
    )

    print(
        "Timing-related outcome uncertainty:",
        bool(row["timing_outcome_uncertain"]),
    )

### Interpretation of the invalid-timing audit

Using the recorded admission timestamps, questionable records were found
within the 30-day windows of 37 index admissions.

Of these:
- 14 also had a valid return within 30 days, so positive evidence remains.
- 23 had no valid return within 30 days and were flagged as having
  timing-related outcome uncertainty. These involve 23 unique patients.

The 23 uncertain outcomes should not be automatically labeled negative.
For the primary analysis, we will exclude these index observations while
preserving their records and audit flags.

This check does not validate or correct the questionable timestamps.
It also does not establish complete follow-up for the remaining admissions.

### Why we retain 14 index admissions and exclude 23

#### What does the audit table count?

The audit table includes all 524,357 potential index admissions.
Each count refers to an index stay whose future outcome we are assessing.

The 175 invalid-timing records come from the full admission history.
They are separate records used to check whether an index admission's
30-day outcome might be uncertain.

For each index stay A, we ask:

1. Is there a valid candidate return within 30 days after A's discharge?
2. Is there also a questionable record within that same window?

Here, a valid candidate has nonmissing admission and discharge times,
with discharge at or after admission. A questionable record fails
the timestamp requirements.

| Valid return within 30 days | Questionable record within 30 days | Index admissions | Treatment |
|---|---|---:|---|
| No | No | 418,219 | Follow-up assessment still required |
| No | Yes | 23 | Exclude because the outcome is uncertain |
| Yes | No | 106,101 | Positive evidence available |
| Yes | Yes | 14 | Retain positive evidence |

#### Why exclude the 23?

These index admissions have a questionable record within 30 days,
but no valid candidate return within that window.

For example:

| Record | Admission | Discharge |
|---|---|---|
| Index stay A | January 1 | January 5 |
| Questionable stay B | January 15 | January 14 |

B's recorded admission occurs 10 days after A's discharge.
However, B's discharge precedes its admission, so B fails our
candidate-return timestamp rule.

This creates two possibilities:

- B represents a real return, but one or more timestamps are incorrect.
- B does not represent a qualifying return under our study definition.

The available records do not resolve which interpretation is correct.

Assigning A a positive label would rely on an invalid-timing record.
Assigning A a negative label would ignore possible evidence of a return.

Therefore, we exclude A from the primary labeled sample rather than
force an uncertain outcome into either class.

Importantly, A itself has valid timestamps. It is excluded because its
outcome is uncertain, not because its own admission record is invalid.

#### Why retain the 14?

These index admissions have both a valid return and a questionable
record within 30 days.

For example:

| Record | Admission | Discharge |
|---|---|---|
| Index stay A | January 1 | January 5 |
| Valid return C | January 10 | January 12 |
| Questionable stay B | January 20 | January 19 |

C begins 5 days after A's discharge and passes our timestamp checks.
Therefore, A already has valid evidence of a return within 30 days.

B remains questionable, but resolving B is unnecessary to answer
our binary outcome question:

“Was at least one qualifying return recorded within 30 days?”

The answer is already yes because of C. We therefore retain A
with positive outcome evidence.

The same reasoning applies if the questionable record occurs before
the valid return: at least one valid return inside the window is
sufficient for the binary outcome.

#### Binary outcome versus exact time to first return

For these 14 admissions, the questionable record could affect our
understanding of the exact first return time. The earliest valid
candidate is not necessarily the true first return.

However, our current outcome is binary: whether any qualifying return
was recorded within 30 days. It does not require knowing the exact
time of the true first return.

A time-to-event analysis would require separate consideration of
this timing uncertainty.

#### Study decision and limitation

Our primary rule excludes index admissions whose binary outcomes
depend on unresolved invalid-timing records.

It does not exclude every index admission associated with a
questionable future record.

Accordingly:

- Exclude 23 index admissions with uncertain outcomes.
- Retain the 14 with independently supported positive outcomes.
- Preserve the questionable source records and audit flags.
- Do not correct timestamps without supporting evidence.

This is a conservative methodological choice, not proof that the
questionable admissions did or did not occur.

After applying this exclusion, 524,334 potential index admissions
would remain. Final sample eligibility still depends on the
follow-up assessment.

### 3.12 Exclude index admissions with uncertain outcomes

We exclude index admissions that have a questionable record within
30 days after discharge but no valid return within that window.

This removes 23 index observations whose binary outcomes cannot be
confidently assigned under our timestamp rules.

The 14 admissions with both a valid return and a questionable record
remain eligible because their positive outcomes are independently supported.

Excluded observations are preserved separately for audit. The full
source admission history remains unchanged.

The remaining sample is still provisional pending follow-up assessment.
No final negative labels are assigned here.

In [ ]:
# Preserve the uncertain index observations for audit.
excluded_timing_outcomes = timing_audit.loc[
    timing_audit["timing_outcome_uncertain"]
].copy()

# Create the remaining potential analysis sample.
# Keep timing_audit unchanged so the exclusion can be reviewed.
cohort_after_timing_review = timing_audit.loc[
    ~timing_audit["timing_outcome_uncertain"]
].copy()


# Verify that all observations are accounted for.
assert (
    len(cohort_after_timing_review)
    + len(excluded_timing_outcomes)
    == len(timing_audit)
)

assert cohort_after_timing_review["hadm_id"].is_unique

# All excluded observations must lack a valid return within 30 days.
assert not excluded_timing_outcomes[
    "observed_return_within_30d"
].any()

# Confirm that no observed positives were lost.
assert (
    cohort_after_timing_review["observed_return_within_30d"].sum()
    == timing_audit["observed_return_within_30d"].sum()
)


# Display aggregate counts only.
print("Index admissions before exclusion:", len(timing_audit))
print("Excluded for uncertain outcomes:", len(excluded_timing_outcomes))
print("Remaining potential index admissions:", len(cohort_after_timing_review))

print(
    "Remaining unique patients:",
    cohort_after_timing_review["subject_id"].nunique(),
)

print(
    "Observed returns within 30 days:",
    cohort_after_timing_review["observed_return_within_30d"].sum(),
)

print(
    "No valid return observed within 30 days; follow-up pending:",
    (~cohort_after_timing_review["observed_return_within_30d"]).sum(),
)

Index admissions before exclusion: 524357
Excluded for uncertain outcomes: 23
Remaining potential index admissions: 524334
Remaining unique patients: 216043
Observed returns within 30 days: 106115
No valid return observed within 30 days; follow-up pending: 418219


### 3.13 Follow-up limitations and interpretation of negative outcomes

#### What the available records establish

A valid admission recorded within 30 days after index discharge provides
positive evidence of a return.

When no qualifying return is found, the records establish only that
no qualifying return was observed in the available admissions history.
They do not establish that the patient remained free of readmission
at every hospital.

#### Why we cannot use a single maximum date

MIMIC-IV shifts dates independently for each patient while preserving
time intervals within that patient.

Consequently, the maximum admission or discharge date across all patients
is not a common observation end date. We do not use it to certify
30-day follow-up.

A patient's last recorded encounter also does not establish the end
of observation. Absence of later encounters may simply mean that the
patient did not return.

#### Primary outcome interpretation

For this retrospective analysis, the outcome is a qualifying return
recorded in the available admissions data within 30 days after discharge.

Under this observed-record outcome definition:

- Y = 1: at least one valid qualifying return was recorded within 30 days.
- Y = 0: no valid qualifying return was recorded within 30 days,
  after applying the outcome-quality exclusions.

A negative label does not certify complete follow-up or prove that
no readmission occurred elsewhere.

#### Remaining limitations

Potential administrative right censoring and admissions to other
hospitals may cause actual returns to be absent from the available data.
These limitations cannot be eliminated by treating missing return
timestamps as confirmed complete follow-up.

We retain admissions with no subsequent recorded return rather than
requiring a future encounter for inclusion. Requiring future encounters
would select patients according to their subsequent healthcare use.

The analysis therefore predicts recorded returns under the available
data coverage, rather than fully ascertained readmission at any hospital.

#### Additional outcome assessment

Post-discharge death requires a separate, explicit decision because
death may occur before a possible return. The mortality follow-up
assessment will be documented before finalizing labels.

### 3.14 Audit recorded deaths after index discharge

A patient may die after discharge before having a return admission.
Death is a competing event because it prevents a subsequent readmission.

We merge the patient-level date of death (`dod`) onto the remaining
potential index admissions using `subject_id`.

We examine recorded deaths on the discharge date through calendar day 30.
This is a calendar-date screen, not an exact 720-hour mortality measure.
We do not assume that a date of death provides a precise death time.

We separately identify:
- Death dates before the index discharge date.
- Death dates within the screened window.
- Whether a valid return within 30 days was also observed.

Missing `dod` means no death date is available in this field; it is not
proof of complete survival follow-up.

No deaths are automatically excluded in this diagnostic. We will use
the results to document how death is handled in the outcome definition.

Date of death is used for outcome assessment only, not as a predictor.

In [ ]:
# Read patient-level death information.
# Use the same raw-data directory as admissions.csv.
patients_path = ADMISSIONS_FILE.parent / "patients.csv"

patient_deaths = pd.read_csv(
    patients_path,
    usecols=["subject_id", "dod"],
)

assert patient_deaths["subject_id"].notna().all()
assert patient_deaths["subject_id"].is_unique

# Check parsing without treating genuinely missing death dates as errors.
raw_dod = patient_deaths["dod"]
parsed_dod = pd.to_datetime(raw_dod, errors="coerce")

assert not (raw_dod.notna() & parsed_dod.isna()).any(), (
    "Some recorded death dates could not be parsed."
)

patient_deaths["dod"] = parsed_dod

# Merge onto the sample remaining after the 23 timing exclusions.
mortality_audit = cohort_after_timing_review.merge(
    patient_deaths,
    on="subject_id",
    how="left",
    validate="many_to_one",
    indicator="_death_merge",
)

assert len(mortality_audit) == len(cohort_after_timing_review)
assert mortality_audit["_death_merge"].eq("both").all()

mortality_audit = mortality_audit.drop(columns="_death_merge")

# Compare calendar dates rather than assuming an exact death time.
death_date = mortality_audit["dod"].dt.normalize()
discharge_date = mortality_audit["dischtime"].dt.normalize()

mortality_audit["death_before_discharge_date"] = (
    death_date.lt(discharge_date)
)

mortality_audit["death_on_discharge_date"] = (
    death_date.eq(discharge_date)
)

mortality_audit["death_date_in_30d_screen"] = (
    death_date.ge(discharge_date)
    & death_date.le(discharge_date + pd.Timedelta(days=30))
)

print("Potential index admissions:", len(mortality_audit))

print(
    "Recorded death date before index discharge date:",
    mortality_audit["death_before_discharge_date"].sum(),
)

print(
    "Recorded death on index discharge date:",
    mortality_audit["death_on_discharge_date"].sum(),
)

print("\nDeath-date screen versus observed return:")
display(
    pd.crosstab(
        mortality_audit["observed_return_within_30d"],
        mortality_audit["death_date_in_30d_screen"],
    )
    .reindex(
        index=[False, True],
        columns=[False, True],
        fill_value=0,
    )
    .rename_axis(
        index="Valid return within 30 days",
        columns="Death date in discharge-day-to-day-30 screen",
    )
)

Potential index admissions: 524334
Recorded death date before index discharge date: 24
Recorded death on index discharge date: 46

Death-date screen versus observed return:


Death date in discharge-day-to-day-30 screen,False,True
Valid return within 30 days,,
False,412954,5265
True,102630,3485


### Interpretation of the mortality audit

#### Patient-level death date versus admission-level death status

`hospital_expire_flag` describes whether the patient died during a
specific admission. In contrast, `dod` is the patient's recorded date
of death and may occur after that admission ends.

Therefore, `hospital_expire_flag = 0` with a nonmissing `dod` is not
automatically inconsistent. A patient can survive an index discharge
and die later, either outside the hospital or during a subsequent stay.

Similarly, discharge to home or hospice followed by a later death date
does not, by itself, indicate a recording inconsistency.

#### Findings in the current sample

The audit includes 524,334 potential index admissions.

| Valid return within 30 days | Death date in discharge-day-to-day-30 screen | Index admissions |
|---|---|---:|
| No | No | 412,954 |
| No | Yes | 5,265 |
| Yes | No | 102,630 |
| Yes | Yes | 3,485 |

The death-date screen includes the index discharge date through calendar
day 30. It is not an exact 720-hour mortality window.

- **5,265 admissions:** no valid return within 30 days, with a recorded
  death date in the screened window. These require an explicit policy
  for handling death before readmission.

- **3,485 admissions:** both a valid return within 30 days and a death
  date in the screened window. We must examine their order. A return
  occurring before death can still support a positive readmission label.

- **24 admissions:** the recorded death date precedes the index discharge
  date. This is a chronological discrepancy requiring review; we cannot
  assume which field is incorrect. These cases fall outside the screen's
  “True” column.

- **46 admissions:** the recorded death date equals the index discharge
  date. These are already included in the screen's “True” column.
  Calendar dates alone cannot establish whether death occurred before
  or after discharge that day.

A “False” death-screen flag means no recorded death date falls inside
the screened interval. It does not necessarily mean that `dod` is missing
or that complete survival follow-up has been established.

#### Next decision

We will examine the ordering of discharge, return admission, and recorded
death before deciding on additional exclusions or final labels.

Death after discharge does not automatically invalidate an index admission.
Death after a qualifying return does not erase that return.

No additional admissions have been excluded in this mortality audit.

### 3.15 Review the order of discharge, return, and death

The mortality screen identified admissions with both a recorded return
and a death date. We now compare their calendar dates.

Among admissions with an observed return within 30 days:

- Return date before death date: the recorded order supports a return
  before death.
- Return date equal to death date: the dates alone cannot establish
  the within-day order.
- Return date after death date: the recorded chronology requires review.

This comparison uses every available death date, not only deaths inside
the initial 30-day mortality screen.

We separately summarize deaths recorded before or on the index discharge
date. These groups may overlap with the return/death findings and must
not be added together as separate exclusions.

This is a diagnostic step. No records are removed or corrected.

In [ ]:
# Work on a copy; preserve the previous mortality audit.
mortality_order_review = mortality_audit.copy()

death_date = mortality_order_review["dod"].dt.normalize()
discharge_date = mortality_order_review["dischtime"].dt.normalize()
return_date = mortality_order_review["return_admittime"].dt.normalize()

has_death = death_date.notna()
has_30d_return = mortality_order_review[
    "observed_return_within_30d"
]

# Classify the recorded order for observed 30-day returns.
mortality_order_review["return_death_order"] = (
    "No observed return within 30 days"
)

mortality_order_review.loc[
    has_30d_return & ~has_death,
    "return_death_order",
] = "Return observed; death date unavailable"

mortality_order_review.loc[
    has_30d_return & has_death & return_date.lt(death_date),
    "return_death_order",
] = "Return date before death date"

mortality_order_review.loc[
    has_30d_return & has_death & return_date.eq(death_date),
    "return_death_order",
] = "Return and death on same date"

mortality_order_review.loc[
    has_30d_return & has_death & return_date.gt(death_date),
    "return_death_order",
] = "Return date after death date"


# Classify death dates relative to the index discharge.
mortality_order_review["death_discharge_order"] = (
    "Death date unavailable"
)

mortality_order_review.loc[
    has_death & death_date.lt(discharge_date),
    "death_discharge_order",
] = "Death date before discharge date"

mortality_order_review.loc[
    has_death & death_date.eq(discharge_date),
    "death_discharge_order",
] = "Death on discharge date"

mortality_order_review.loc[
    has_death & death_date.gt(discharge_date),
    "death_discharge_order",
] = "Death date after discharge date"


print("Order of return and death among observed 30-day returns:")
display(
    mortality_order_review.loc[
        has_30d_return, "return_death_order"
    ]
    .value_counts()
    .rename_axis("recorded_order")
    .to_frame("index_admissions")
)

print("\nDeath date relative to index discharge:")
display(
    pd.crosstab(
        mortality_order_review["death_discharge_order"],
        has_30d_return,
    )
    .reindex(columns=[False, True], fill_value=0)
    .rename_axis(
        columns="Valid return within 30 days"
    )
)

print("\nReturn/death order among the 3,485 screen-positive returns:")
display(
    mortality_order_review.loc[
        has_30d_return
        & mortality_order_review["death_date_in_30d_screen"],
        "return_death_order",
    ]
    .value_counts()
    .rename_axis("recorded_order")
    .to_frame("index_admissions")
)

# Verify the classification preserves all observed positive cases.
assert (
    mortality_order_review.loc[
        has_30d_return, "return_death_order"
    ].value_counts().sum()
    == 106115
)

Order of return and death among observed 30-day returns:


,index_admissions
recorded_order,
Return observed; death date unavailable,65522
Return date before death date,40513
Return and death on same date,77
Return date after death date,3



Death date relative to index discharge:


Valid return within 30 days,False,True
death_discharge_order,,
Death date after discharge date,87070,40590
Death date before discharge date,21,3
Death date unavailable,331082,65522
Death on discharge date,46,0



Return/death order among the 3,485 screen-positive returns:


,index_admissions
recorded_order,
Return date before death date,3408
Return and death on same date,77


### Interpretation of event ordering

Among 106,115 index admissions with an observed return within 30 days,
40,513 have a return date before the recorded death date, 77 have a return
on the death date, and 3 have a return date after death. The remaining
65,522 have no available patient-level death date.

The 3 returns recorded after death are also among the 24 index admissions
with death recorded before index discharge. These findings overlap and
must not be counted as separate exclusions.

Among the 3,485 returns with death in the initial mortality screen:
- 3,408 have a return date before death.
- 77 have return and death on the same calendar date.

Same-day return and death can represent a genuine readmission followed
by death. Calendar dates alone do not establish the within-day order;
the return admission's mortality fields require further review.

All 46 deaths recorded on the index discharge date occur among admissions
without an observed 30-day return.

No additional exclusions are applied here. Next, we will inspect the
mortality fields associated with the chronological discrepancies and
same-day cases before documenting the final handling rules.

### 3.16 Compare mortality fields for flagged records

We attach the matched return admission's discharge and mortality fields
from the full source admission history.

For returns occurring on the recorded death date, we check whether
the return admission records a death time at or after its admission time.
Such a timestamp supports the recorded sequence of return followed by death.

We also summarize the index admission's mortality fields for:
- Death dates before index discharge.
- Death dates equal to index discharge.

These comparisons provide supporting evidence; they do not establish
which source is correct when fields disagree.

No timestamps are corrected and no additional exclusions are applied.

In [ ]:
# Attach details of the matched return admission.
return_details = admission_history[
    [
        "hadm_id",
        "dischtime",
        "deathtime",
        "hospital_expire_flag",
        "discharge_location",
    ]
].rename(columns={
    "hadm_id": "return_hadm_id",
    "dischtime": "return_dischtime",
    "deathtime": "return_deathtime",
    "hospital_expire_flag": "return_expire_flag",
    "discharge_location": "return_discharge_location",
})

return_details["return_hadm_id"] = (
    return_details["return_hadm_id"].astype("Int64")
)

mortality_detail_review = mortality_order_review.merge(
    return_details,
    on="return_hadm_id",
    how="left",
    validate="many_to_one",
)

assert len(mortality_detail_review) == len(mortality_order_review)

# Select the 77 index admissions with return and death on the same date.
same_day_returns = mortality_detail_review.loc[
    mortality_detail_review["return_death_order"].eq(
        "Return and death on same date"
    )
].copy()

# Describe the return admission's recorded death timestamp.
same_day_returns["return_death_time_evidence"] = (
    "No return death timestamp"
)

has_time = same_day_returns["return_deathtime"].notna()

same_day_returns.loc[
    has_time
    & same_day_returns["return_deathtime"].lt(
        same_day_returns["return_admittime"]
    ),
    "return_death_time_evidence",
] = "Death timestamp before return admission"

same_day_returns.loc[
    has_time
    & same_day_returns["return_deathtime"].ge(
        same_day_returns["return_admittime"]
    ),
    "return_death_time_evidence",
] = "Death timestamp at or after return admission"

print("Same-day return/death: timestamp evidence")
display(
    same_day_returns["return_death_time_evidence"]
    .value_counts()
    .to_frame("index_admissions")
)

print("\nSame-day return/death: return disposition and death flag")
display(
    same_day_returns.groupby(
        ["return_expire_flag", "return_discharge_location"],
        dropna=False,
    )
    .size()
    .to_frame("index_admissions")
)

# Check whether the return death timestamp agrees with patient-level dod.
same_day_returns["return_deathtime_date_matches_dod"] = (
    same_day_returns["return_deathtime"].dt.normalize()
    .eq(same_day_returns["dod"].dt.normalize())
)

print("\nSame-day return/death: death timestamp date agrees with dod")
display(
    pd.crosstab(
        same_day_returns["return_death_time_evidence"],
        same_day_returns["return_deathtime_date_matches_dod"],
    )
)


# Review index mortality fields for death before/on discharge date.
index_death_flags = mortality_detail_review.loc[
    mortality_detail_review["death_discharge_order"].isin([
        "Death date before discharge date",
        "Death on discharge date",
    ])
].copy()

index_death_flags["index_deathtime_present"] = (
    index_death_flags["deathtime"].notna()
)

print("\nFlagged index admissions: index mortality fields")
display(
    index_death_flags.groupby(
        [
            "death_discharge_order",
            "hospital_expire_flag",
            "discharge_location",
            "index_deathtime_present",
        ],
        dropna=False,
    )
    .size()
    .to_frame("index_admissions")
)

Same-day return/death: timestamp evidence


,index_admissions
return_death_time_evidence,
Death timestamp at or after return admission,76
Death timestamp before return admission,1



Same-day return/death: return disposition and death flag


index_admissions
return_expire_flag return_discharge_location                  
1.0                DIED                                     65
                   HOME                                      1
                   SKILLED NURSING FACILITY                  1
                   NaN                                      10


Same-day return/death: death timestamp date agrees with dod


return_deathtime_date_matches_dod,True
return_death_time_evidence,
Death timestamp at or after return admission,76
Death timestamp before return admission,1



Flagged index admissions: index mortality fields


index_admissions
death_discharge_order            hospital_expire_flag discharge_location           index_deathtime_present                  
Death date before discharge date 0                    CHRONIC/LONG TERM ACUTE CARE False                                   1
                                                      HOME                         False                                   7
                                                      HOME HEALTH CARE             False                                   6
                                                      SKILLED NURSING FACILITY     False                                   4
                                                      NaN                          False                                   6
Death on discharge date          0                    ACUTE HOSPITAL               False                                   3
                                                      AGAINST ADVICE               False                                   1
                                                      HOME                         False                                   8
                                                      HOME HEALTH CARE             False                                   8
                                                      REHAB                        False                                   2
                                                      SKILLED NURSING FACILITY     False                                  15
                                                      NaN                          False                                   9

### Interpretation of mortality-field comparisons

Among the 77 index admissions with return and death on the same date,
76 have a return admission death timestamp at or after return admission.
This supports retaining their positive return evidence, subject to the
remaining chronology checks.

One has a death timestamp before return admission and requires further
review. We do not correct its timestamps or finalize its label here.

All 77 return admissions have hospital_expire_flag = 1, and their
death timestamp dates agree with patient-level dod. Agreement at the
date level does not resolve the one conflicting within-day sequence.

The 24 index admissions with dod before discharge all have an index
expire flag of 0 and no index deathtime. These discrepancies remain
unresolved; the available fields do not establish which source is wrong.

The 46 index admissions with death on the discharge date also have
an index expire flag of 0 and no index deathtime. They are not confirmed
in-hospital deaths, and the date alone cannot establish the event order.

No additional exclusions have been applied.

### 3.17 Check death timestamps against the return stay

For the 77 returns occurring on the patient’s recorded death date,
we compare the return admission's death timestamp with its admission
and discharge timestamps.

We distinguish:
- Death timestamp before return admission.
- Death timestamp within the return stay, including its boundaries.
- Death timestamp after return discharge.
- Missing death timestamp.

These comparisons describe recorded chronology. They do not establish
which timestamp is correct when a discrepancy exists.

No records are corrected or excluded in this step.

In [ ]:
# Review the same 77 cases from Section 3.16.
same_day_timing_review = same_day_returns.copy()

death_time = same_day_timing_review["return_deathtime"]
admit_time = same_day_timing_review["return_admittime"]
discharge_time = same_day_timing_review["return_dischtime"]

# Assign mutually exclusive timing categories.
same_day_timing_review["death_timestamp_position"] = (
    "Missing death timestamp"
)

same_day_timing_review.loc[
    death_time.notna() & death_time.lt(admit_time),
    "death_timestamp_position",
] = "Before return admission"

same_day_timing_review.loc[
    death_time.notna()
    & death_time.ge(admit_time)
    & death_time.le(discharge_time),
    "death_timestamp_position",
] = "Within return stay"

same_day_timing_review.loc[
    death_time.notna() & death_time.gt(discharge_time),
    "death_timestamp_position",
] = "After return discharge"

# Calculate signed differences.
# Negative first interval means death was recorded before admission.
same_day_timing_review["hours_admission_to_death"] = (
    (death_time - admit_time).dt.total_seconds() / 3600
)

# Negative second interval means death was recorded after discharge.
same_day_timing_review["hours_death_to_discharge"] = (
    (discharge_time - death_time).dt.total_seconds() / 3600
)

print("Death timestamp position:")
display(
    same_day_timing_review["death_timestamp_position"]
    .value_counts()
    .to_frame("index_admissions")
)

# Display only problematic cases, with aliases and relative intervals.
# No actual IDs or dates are printed.
problem_cases = same_day_timing_review.loc[
    ~same_day_timing_review["death_timestamp_position"].eq(
        "Within return stay"
    ),
    [
        "death_timestamp_position",
        "hours_admission_to_death",
        "hours_death_to_discharge",
        "return_expire_flag",
        "return_discharge_location",
    ],
].copy()

problem_cases = problem_cases.reset_index(drop=True)

problem_cases.insert(
    0,
    "example",
    [f"Case_{i:02d}" for i in range(1, len(problem_cases) + 1)],
)

print("\nCases requiring review:")
display(problem_cases.round(3))

assert len(same_day_timing_review) == 77

### Interpretation of the return death-timestamp check

Of the 77 same-day return/death cases, 73 have death timestamps within
the return stay, supporting the recorded sequence of return followed
by death.

Three have death timestamps after return discharge. These records
contain mortality/discharge discrepancies, but their admission times
precede the recorded death times. The discrepancies therefore do not,
by themselves, negate the observed return.

One has a death timestamp 14.75 hours before return admission.
Its chronology remains unresolved and its positive label requires
separate consideration.

No timestamps have been corrected and no additional exclusions have
been applied. These findings distinguish uncertainty about whether
a return preceded death from discrepancies occurring after a return.

### 3.18 Apply mortality-handling rules

Our outcome is a qualifying return recorded within 30 days after
index discharge. We do not require survival throughout the window.

#### Chronological inconsistencies

For the primary analysis, exclude index observations with:
- Patient-level death date before index discharge date.
- A same-day matched return whose recorded death timestamp precedes
  its admission timestamp.

These exclusions reflect unresolved chronology. We preserve the
original records and do not assume which timestamp is incorrect.

#### Returns followed by death

Retain positive return evidence when return admission precedes the
recorded death. Death after a return does not erase that return.

The three cases with death timestamps after return discharge remain
flagged as discrepancies, but this does not by itself negate their
recorded admission before death.

#### Death without an observed return

Retain otherwise eligible index admissions with post-discharge death
and no qualifying return as negative for the recorded-return outcome.

Death is a competing event. These negatives must not be interpreted
as patients who remained alive and readmission-free for 30 days.

The 46 deaths on the index discharge date remain flagged because
calendar dates cannot establish the within-day order. They are retained
under the observed-record outcome definition.

Mortality audit fields are not model predictors. This policy does not
resolve administrative censoring or returns outside the dataset.

In [ ]:
# Identify the same-day return with death recorded before admission.
uncertain_return_ids = same_day_timing_review.loc[
    same_day_timing_review["death_timestamp_position"].eq(
        "Before return admission"
    ),
    "hadm_id",
]

# Apply the policy to a copy of the full mortality review.
mortality_policy_audit = mortality_detail_review.copy()

mortality_policy_audit["exclude_death_before_discharge"] = (
    mortality_policy_audit["death_before_discharge_date"]
)

mortality_policy_audit["exclude_uncertain_return_chronology"] = (
    mortality_policy_audit["hadm_id"].isin(uncertain_return_ids)
)

# Use a union so an admission cannot be excluded twice.
mortality_policy_audit["exclude_mortality_chronology"] = (
    mortality_policy_audit["exclude_death_before_discharge"]
    | mortality_policy_audit["exclude_uncertain_return_chronology"]
)

# Preserve excluded index observations separately.
excluded_mortality_chronology = mortality_policy_audit.loc[
    mortality_policy_audit["exclude_mortality_chronology"]
].copy()

cohort_after_mortality_review = mortality_policy_audit.loc[
    ~mortality_policy_audit["exclude_mortality_chronology"]
].copy()

# Preserve a descriptive competing-death flag.
# This uses the calendar-date screen, not exact death timing.
cohort_after_mortality_review["death_screen_without_30d_return"] = (
    cohort_after_mortality_review["death_date_in_30d_screen"]
    & ~cohort_after_mortality_review["observed_return_within_30d"]
)

# Verify the exclusion preserves one row per remaining index admission.
assert cohort_after_mortality_review["hadm_id"].is_unique
assert (
    len(cohort_after_mortality_review)
    + len(excluded_mortality_chronology)
    == len(mortality_policy_audit)
)

print("Before mortality exclusions:", len(mortality_policy_audit))
print("Excluded for mortality chronology:",
      len(excluded_mortality_chronology))
print("Remaining index admissions:",
      len(cohort_after_mortality_review))

print("\nObserved return status after exclusions:")
display(
    cohort_after_mortality_review[
        "observed_return_within_30d"
    ].value_counts().to_frame("index_admissions")
)

print(
    "\nDeath-screen cases without a 30-day return retained:",
    cohort_after_mortality_review[
        "death_screen_without_30d_return"
    ].sum(),
)

Before mortality exclusions: 524334
Excluded for mortality chronology: 25
Remaining index admissions: 524309

Observed return status after exclusions:


,index_admissions
observed_return_within_30d,
False,418198
True,106111



Death-screen cases without a 30-day return retained: 5265


### Mortality-policy validation summary

The mortality chronology rule excluded 25 index observations:
24 with a death date before index discharge and one with a death
timestamp before the matched same-day return admission.

The remaining sample contains 524,309 index admissions:
- 106,111 with an observed qualifying return within 30 days.
- 418,198 without an observed qualifying return within 30 days.

The 5,265 admissions with death in the calendar-date screen and no
qualifying return remain included under the recorded-return definition.
They are not interpreted as patients who survived the full window.

The original records and mortality audit information remain preserved.

### 3.19 Finalize the recorded-return label

After applying the documented exclusions:

- Y = 1: a qualifying admission was recorded strictly after index
  discharge and within 30 days.
- Y = 0: no qualifying admission was recorded within that window.

Negative labels do not certify complete follow-up, survival for 30 days,
or absence of readmission at another hospital.

We prepare two tables:
1. A labeled cohort containing the original Step 2 columns and outcome.
2. A separate audit table containing future-return and mortality details.

Future-event audit fields must not enter model predictors. Feature
selection and availability checks will be completed in Step 4.

In [ ]:
# Assign the label under our documented recorded-return definition.
final_label_review = cohort_after_mortality_review.copy()

final_label_review["label_readmit_30d"] = (
    final_label_review["observed_return_within_30d"]
    .astype("int8")
)

# Independently verify labels against exact intervals.
expected_positive = (
    final_label_review["gap_to_return"].gt(pd.Timedelta(0))
    & final_label_review["gap_to_return"].le(
        pd.Timedelta(days=30)
    )
)

assert final_label_review["label_readmit_30d"].eq(
    expected_positive.astype("int8")
).all()

assert final_label_review["hadm_id"].is_unique
assert final_label_review["label_readmit_30d"].notna().all()
assert final_label_review["label_readmit_30d"].isin([0, 1]).all()

assert not final_label_review["timing_outcome_uncertain"].any()
assert not final_label_review["exclude_mortality_chronology"].any()


# Prepare the downstream cohort using the original Step 2 schema.
# This prevents newly added future-event audit columns from
# accidentally entering feature engineering.
step2_columns = index_cohort.columns.tolist()

assert "label_readmit_30d" not in step2_columns

cohort_step3 = (
    final_label_review[
        step2_columns + ["label_readmit_30d"]
    ]
    .sort_values(["subject_id", "admittime", "hadm_id"])
    .reset_index(drop=True)
)

# Preserve the full audit information separately.
label_audit_step3 = (
    final_label_review
    .sort_values(["subject_id", "admittime", "hadm_id"])
    .reset_index(drop=True)
)


# Summarize the labels.
label_summary_step3 = (
    cohort_step3["label_readmit_30d"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("label_readmit_30d")
    .to_frame("admissions")
)

label_summary_step3["percent"] = (
    label_summary_step3["admissions"]
    .div(len(cohort_step3))
    .mul(100)
    .round(2)
)

print("Labeled index admissions:", len(cohort_step3))
print("Unique patients:", cohort_step3["subject_id"].nunique())
print("Labeled cohort columns:", len(cohort_step3.columns))

display(label_summary_step3)


# Record the sequential Step 3 exclusions.
step3_flow = pd.DataFrame({
    "stage": [
        "Step 2 eligible index admissions",
        "After overlapping/touching index exclusion",
        "After uncertain-outcome exclusion",
        "After mortality-chronology exclusion",
    ],
    "remaining_admissions": [
        len(index_cohort),
        len(index_for_labeling),
        len(cohort_after_timing_review),
        len(cohort_step3),
    ],
    "excluded_at_stage": [
        0,
        len(ambiguous_index_ids),
        len(excluded_timing_outcomes),
        len(excluded_mortality_chronology),
    ],
})

display(step3_flow)

Labeled index admissions: 524309
Unique patients: 216032
Labeled cohort columns: 20


,admissions,percent
label_readmit_30d,,
0,418198,79.76
1,106111,20.24


,stage,remaining_admissions,excluded_at_stage
0,Step 2 eligible index admissions,528551,0
1,After overlapping/touching index exclusion,524357,4194
2,After uncertain-outcome exclusion,524334,23
3,After mortality-chronology exclusion,524309,25


### 3.20 Export and verify Step 3 outputs

We save:
- The labeled cohort for Step 4 feature engineering.
- A separate table containing future-event audit information.
- The label distribution and sequential exclusion counts.
- The index admissions excluded during Step 3, with exclusion reasons.

The audit and exclusion files support reproducibility and must not
be used as model predictor tables.

After saving, we reload the labeled cohort and verify its columns,
admission/patient identifiers, and labels against the in-memory table.

These files contain patient-level data and remain in the local processed
data directory. Public GitHub materials should contain code and suitable
aggregate summaries.

### Understanding the cohort and label-audit tables

#### Same admissions, different purposes

`cohort_step3.csv` and `label_audit_step3.csv` contain the same final
524,309 eligible index admissions, with one row per `hadm_id`.

The difference is their columns and intended use:

| Table | Contents | Purpose |
|---|---|---|
| `cohort_step3.csv` | Original Step 2 columns plus `label_readmit_30d` | Starting table for Step 4 feature engineering |
| `label_audit_step3.csv` | Cohort columns plus return details, mortality information, and validation flags | Explain and review label construction |

The audit table contains additional information about the same index
admissions. It is not a separate sample of return admissions.

#### What does the cohort table contain?

The cohort contains 20 columns:
- The original 19 Step 2 columns.
- The outcome column, `label_readmit_30d`.

Identifiers connect each index admission to other source tables.
Admission timestamps define the index stay and support feature windows.

The cohort is not yet a model-ready predictor matrix. Step 4 must decide
which columns become features, which require transformation, and which
must be excluded.

`label_readmit_30d` is the target Y, not a predictor in X.
Identifiers are linkage and grouping fields, not model predictors.

#### Additional columns in the audit table

The current pipeline adds the following columns beyond the cohort schema.
They are audit information, not automatically eligible model features.

| Column | Meaning |
|---|---|
| `return_hadm_id` | Admission ID of the earliest valid candidate starting after index discharge |
| `return_admittime` | Admission timestamp of that candidate |
| `return_admission_type` | Admission type of that candidate |
| `gap_to_return` | Exact interval from index discharge to candidate admission |
| `observed_return_within_30d` | Whether the candidate begins strictly after discharge and within 30 days |
| `questionable_hadm_id` | ID of the earliest invalid-timing record starting after index discharge |
| `questionable_admittime` | Recorded admission timestamp of that questionable record |
| `gap_to_questionable` | Interval from index discharge to the questionable admission |
| `questionable_record_within_30d` | Whether that questionable record falls inside the outcome window |
| `timing_outcome_uncertain` | Whether questionable evidence exists without a valid 30-day return |
| `dod` | Patient-level recorded date of death |
| `death_before_discharge_date` | Whether death date precedes index discharge date |
| `death_on_discharge_date` | Whether death and index discharge share a calendar date |
| `death_date_in_30d_screen` | Whether death date falls from discharge day through calendar day 30 |
| `return_death_order` | Recorded calendar-date ordering of return and death |
| `death_discharge_order` | Recorded calendar-date ordering of death and index discharge |
| `return_dischtime` | Discharge timestamp of the matched return admission |
| `return_deathtime` | Death timestamp recorded on the matched return admission |
| `return_expire_flag` | In-hospital death flag for the matched return admission |
| `return_discharge_location` | Disposition recorded for the matched return admission |
| `exclude_death_before_discharge` | Mortality exclusion flag for death before index discharge |
| `exclude_uncertain_return_chronology` | Exclusion flag for the unresolved same-day return chronology |
| `exclude_mortality_chronology` | Combined mortality-chronology exclusion flag |
| `death_screen_without_30d_return` | Death in the calendar-date screen with no observed qualifying return |

Some diagnostic columns were created only in temporary review tables.
For example, `death_timestamp_position` and
`hours_admission_to_death` are not automatically included in the exported
audit unless explicitly merged into it.

#### Example: how the two tables describe the same index admission

Suppose index stay A ends January 1 and the patient's next valid
admission begins January 10.

The cohort records:
- The index admission's original information.
- `label_readmit_30d = 1`.

The audit additionally records:
- Which return admission was matched.
- Its admission time and admission type.
- A gap of nine days.
- The timing and mortality checks associated with that index admission.

The cohort provides the admission and outcome needed downstream.
The audit provides the evidence used to construct and review that outcome.

#### When and how to use each table

**Use the cohort in Step 4**

Load `cohort_step3.csv` and attach appropriately timed features from
other source tables using `hadm_id` and `subject_id`.

Feature engineering must use information available by index discharge.
Any learned preprocessing, such as imputation or scaling, must later
be fitted on training data only.

**Use the audit to investigate labels**

Load `label_audit_step3.csv` when checking questions such as:
- Why did an index admission receive label 1?
- Was its matched return inside the exact 30-day window?
- Was a questionable record also present?
- Did death occur before or after the recorded return?
- Does a future revision of the outcome rule change the label?

Link the audit to the cohort by `hadm_id`. Check that `subject_id` agrees.
Use a one-to-one merge when comparing these final tables.

A matched return may occur after 30 days. Its presence in the audit
does not automatically imply a positive label.

#### Preventing outcome leakage

Future return timestamps, return admission characteristics, death
information, and outcome-derived flags must not enter the predictor
matrix.

For example, `gap_to_return` directly reveals whether the return falls
inside 30 days. Using it as a predictor would give the model information
that was unavailable at index discharge.

Keeping the audit separate reduces accidental leakage, but Step 4 still
requires an explicit review of every proposed predictor.

#### Where are excluded admissions stored?

Both final tables contain retained admissions only.

Therefore, exclusion flags in the final audit should be False for the
exclusions already applied. To identify removed index admissions and
their reasons, use `cohort_step3_exclusions.csv`.

The exclusion ledger records identifiers and reasons; it does not contain
all diagnostic details. Detailed excluded records remain in the separate
in-memory exclusion tables unless exported separately.

#### CSV loading notes

CSV files do not preserve pandas datetime or timedelta data types.

When reloading:
- Parse timestamp columns with `pd.to_datetime`.
- Parse gap columns with `pd.to_timedelta`.
- Preserve missing return timestamps and gaps as missing values.

A negative label means no qualifying return was recorded under the
study rules. It does not prove complete follow-up, survival for 30 days,
or absence of admission at another hospital.

In [ ]:
# Save to the project's processed-data directory.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_paths = {
    "cohort": PROCESSED_DIR / "cohort_step3.csv",
    "audit": PROCESSED_DIR / "label_audit_step3.csv",
    "summary": PROCESSED_DIR / "label_summary_step3.csv",
    "flow": PROCESSED_DIR / "cohort_step3_flow.csv",
    "exclusions": PROCESSED_DIR / "cohort_step3_exclusions.csv",
}


# Prepare one exclusion ledger covering all Step 3 exclusions.
def exclusion_rows(table, reason):
    result = table[["subject_id", "hadm_id"]].copy()
    result["exclusion_reason"] = reason
    return result


excluded_overlap = index_cohort.loc[
    index_cohort["hadm_id"].isin(ambiguous_index_ids)
]

exclusion_ledger_step3 = pd.concat(
    [
        exclusion_rows(
            excluded_overlap,
            "Overlapping or touching admission intervals",
        ),
        exclusion_rows(
            excluded_timing_outcomes,
            "Uncertain outcome from invalid-timing return record",
        ),
        exclusion_rows(
            excluded_mortality_chronology,
            "Unresolved mortality chronology",
        ),
    ],
    ignore_index=True,
)

# The exclusions were applied sequentially and must not overlap.
assert exclusion_ledger_step3["hadm_id"].is_unique

assert (
    len(cohort_step3) + len(exclusion_ledger_step3)
    == len(index_cohort)
)

assert not cohort_step3["hadm_id"].isin(
    exclusion_ledger_step3["hadm_id"]
).any()


# Export the prepared tables.
# Rerunning this cell replaces these same output files.
cohort_step3.to_csv(output_paths["cohort"], index=False)
label_audit_step3.to_csv(output_paths["audit"], index=False)

label_summary_step3.reset_index().to_csv(
    output_paths["summary"], index=False
)

step3_flow.to_csv(output_paths["flow"], index=False)

exclusion_ledger_step3.to_csv(
    output_paths["exclusions"], index=False
)


# Reload the labeled cohort and verify identifiers and labels.
saved_cohort = pd.read_csv(output_paths["cohort"])

assert saved_cohort.columns.tolist() == cohort_step3.columns.tolist()
assert len(saved_cohort) == len(cohort_step3)
assert saved_cohort["hadm_id"].is_unique

verification_columns = [
    "subject_id",
    "hadm_id",
    "label_readmit_30d",
]

pd.testing.assert_frame_equal(
    saved_cohort[verification_columns].reset_index(drop=True),
    cohort_step3[verification_columns].reset_index(drop=True),
    check_dtype=False,
)

# Confirm all output files exist and contain data.
for name, path in output_paths.items():
    assert path.is_file() and path.stat().st_size > 0
    print(f"{name}: {path}")

print("\nSaved labeled cohort verified.")
print("Admissions:", len(saved_cohort))
print("Unique patients:", saved_cohort["subject_id"].nunique())
print("Step 3 excluded admissions:", len(exclusion_ledger_step3))

## Pipeline status — Step 3 completed

### Overall project pipeline

1. **Load and validate data — completed**
   - Load source tables.
   - Check required columns, identifiers, and table relationships.

2. **Construct the index cohort — completed**
   - Apply timestamp, mortality, hospice, and adult eligibility rules.
   - Retain all admission types, including elective admissions.
   - Output: `cohort_step2.csv`.

3. **Construct and validate readmission labels — completed**
   - Use the Step 2 cohort as potential index admissions.
   - Search the full admission history for candidate returns.
   - Review overlapping stays, invalid timestamps, and mortality chronology.
   - Assign the recorded 30-day return outcome.
   - Output: `cohort_step3.csv`.

4. **Build features — NEXT**
   - Start from `cohort_step3.csv`.
   - Attach features using information available by index discharge.
   - Preserve one row per eligible index admission.
   - Keep future-event audit information out of predictors.

5. **Split by patient — pending**
   - Separate training, validation, and test patients.
   - Prevent the same patient from appearing across partitions.

6. **Train and evaluate models — pending**
   - Fit preprocessing using training data only.
   - Compare Logistic Regression, Random Forest, and XGBoost.
   - Evaluate discrimination and calibration.

7. **Interpret and report results — pending**
   - Examine model explanations and limitations.
   - Update results, figures, and project documentation.

---

### Step 3 cohort flow

| Stage | Excluded at this stage | Remaining index admissions |
|---|---:|---:|
| Step 2 eligible index cohort | — | 528,551 |
| Exclude overlapping or touching index stays | 4,194 | 524,357 |
| Exclude uncertain outcomes involving invalid-timing records | 23 | 524,334 |
| Exclude unresolved mortality chronology | 25 | **524,309** |
| **Total Step 3 exclusions** | **4,242** | **524,309** |

Exclusions apply to index observations. The full source admission history
is preserved for outcome assessment.

The 175 invalid-timing source records are not an additional exclusion
from this flow: they were already outside the Step 2 cohort. Reviewing
their effect on earlier index admissions identified the 23 uncertain
outcomes shown above.

---

### Final Step 3 sample and outcome

- **Unit of observation:** one eligible index admission.
- **Prediction point:** index discharge.
- **Final admissions:** 524,309.
- **Unique patients:** 216,032.
- **Outcome window:** strictly after discharge through exactly 30 days.

| Label | Meaning | Admissions | Percentage |
|---|---|---:|---:|
| 0 | No qualifying return recorded within 30 days | 418,198 | 79.76% |
| 1 | At least one qualifying return recorded within 30 days | 106,111 | 20.24% |
| **Total** | | **524,309** | **100%** |

Candidate returns include all admission types represented in the source
admissions table, including elective and observation admissions.
A return ending in death or hospice is not automatically disqualified.

Post-discharge death without a qualifying return remains negative under
the recorded-return definition. Negative labels do not establish
30-day survival, complete follow-up, or absence of readmission elsewhere.

---

### Saved Step 3 outputs

| File | Purpose |
|---|---|
| `cohort_step3.csv` | Final eligible cohort and label; input to Step 4 |
| `label_audit_step3.csv` | Return matching, mortality details, and audit flags for retained admissions |
| `label_summary_step3.csv` | Label counts and percentages |
| `cohort_step3_flow.csv` | Sequential index exclusions and remaining counts |
| `cohort_step3_exclusions.csv` | Excluded index admission identifiers and reasons |

Export verification confirmed the saved cohort's column names,
admission/patient identifiers, and labels against the in-memory table.

---

### Where we are now

**Step 3 is complete. Step 4 feature engineering is next.**

The Step 4 starting table contains the original 19 Step 2 columns plus
`label_readmit_30d`, giving 20 columns.

This is a labeled cohort, not yet a model-ready feature matrix.
Step 4 will define predictors X while preserving the outcome Y.

The revised cohort and labels have not yet been used to rerun the
downstream models. Previous model results must be refreshed before
being reported as results for this sample.